In [ ]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'brfss_xpt', 'LLCP2015XPT (1).zip', 'sample_data']


In [ ]:
print(os.listdir("/content/brfss_xpt"))

[]


In [ ]:
zip_path ="/content/LLCP2015XPT (1).zip"
print("Exists:", os.path.exists(zip_path))
print("Size in bytes:", os.path.getsize(zip_path))

Exists: True
Size in bytes: 98843355


In [ ]:
zip_path ="/content/LLCP2015XPT (1).zip"
with open(zip_path, "rb") as f:
      first_bytes =f.read(20)

print (first_bytes)

b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xd9\x86\x17I;\xfe\xb1\xc5\r:'


In [ ]:
import pandas as pd

try:
  brfss=pd.read_sas("/content/LLCP2015XPT (1).zip", format="xport", encoding="utf-8")
  print("It is actually an XPT file.")
  print(brfss.shape)
  brfss.head()
except Exception as e:
  print("Direct XPT read failed:")
  print(e)

It is actually an XPT file.
(441456, 330)


In [ ]:
brfss=pd.read_sas("/content/LLCP2015XPT (1).zip", format="xport", encoding="utf-8")
print(brfss.shape)
brfss.head()

(441456, 330)


,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENUM,...,_PAREC1,_PASTAE1,_LMTACT1,_LMTWRK1,_LMTSCL1,_RFSEAT2,_RFSEAT3,_FLSHOT6,_PNEUMO2,_AIDTST3
0,1.0,1.0,01292015,01,29,2015,1200.0,2.015000e+09,2.015000e+09,1.0,...,4.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0
1,1.0,1.0,01202015,01,20,2015,1100.0,2.015000e+09,2.015000e+09,1.0,...,2.0,2.0,3.0,3.0,4.0,2.0,2.0,NaN,NaN,2.0
2,1.0,1.0,02012015,02,01,2015,1200.0,2.015000e+09,2.015000e+09,1.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,NaN
3,1.0,1.0,01142015,01,14,2015,1100.0,2.015000e+09,2.015000e+09,1.0,...,4.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,9.0
4,1.0,1.0,01142015,01,14,2015,1100.0,2.015000e+09,2.015000e+09,1.0,...,4.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0


In [ ]:
#creating a subset of BRFSS
brfss_sub =brfss[[
    "DIABETE3",
    "SEX",
    "_AGE80",
    "_BMI5",
    "BPHIGH4",
    "_RFSMOK3",
    "_TOTINDA"
]].copy()

print(brfss_sub.shape)
brfss_sub.head()

(441456, 7)


,DIABETE3,SEX,_AGE80,_BMI5,BPHIGH4,_RFSMOK3,_TOTINDA
0,3.0,2.0,63.0,4018.0,1.0,1.0,2.0
1,3.0,2.0,52.0,2509.0,3.0,2.0,1.0
2,3.0,2.0,71.0,2204.0,3.0,9.0,9.0
3,3.0,2.0,63.0,2819.0,1.0,1.0,2.0
4,3.0,2.0,61.0,2437.0,3.0,1.0,2.0


In [ ]:
#Recoding BRFSS

#Keeping only clear diabetes outcome cases
brfss_sub =brfss_sub[brfss_sub["DIABETE3"].isin([1.0,3.0])].copy()
brfss_sub["diabetes_binary"]= brfss_sub["DIABETE3"].map({1.0: 1, 3.0: 0})

#Sex: 1=male, 2=female
brfss_sub["sex_male"]= brfss_sub["SEX"].map({1.0: 1, 2.0: 0})

#Age
brfss_sub["age"] =brfss_sub["_AGE80"]

#BMI: two implied decimals
brfss_sub["bmi"] =brfss_sub["_BMI5"] /100.0

#Hypertension: 1=yes, 4= told borderline high, 3=no, 2= only during preganancy
brfss_sub["hypertension"] =np.where(
    brfss_sub["BPHIGH4"].isin([1.0, 4.0]), 1,
    np.where(brfss_sub["BPHIGH4"] ==3.0, 0, np.nan)
)

#Current Smoker: 2=current smoker, 1=not current smoker
brfss_sub["current_smoker"] =np.where(
    brfss_sub["_RFSMOK3"] ==2.0, 1,
    np.where(brfss_sub["_RFSMOK3"]==1.0, 0, np.nan)
)


#Physically active: 1=active, 2=inactive
brfss_sub["physically_active"] =np.where(
    brfss_sub["_TOTINDA"] ==1.0, 1,
    np.where(brfss_sub["_TOTINDA"] ==2.0, 0, np.nan)
)


# **So in relation to the Codebook we have**:

**DIABETE3**, we have:

1.0 =diabetes

3.0 =no diabetes

2.0 =pregnancy-related only

4.0 =prediabetes/borderline

7.0, 9.0 =unclear/refused

*So for my thesis, i need to keep only 1.0 and 3.0  and drop the rest*


**SEX, we have:**

1.0 =male
2.0 =female

**_AGE80,** this is a usable numeric age variable so I will keep it directly as age.

**_BMI5** with 2 implied decimals so we divide by 100 and missing values are already NaN

**BPHIGH4, we have:**

1.0 =yes, high blood pressure

2.0 =yes, but only during pregnancy

3.0 =no

4.0 =told borderline high

7.0, 9.0 = unclear/refused

So for the harmonized variables, I will use:

1.0 and 4.0 (hypertension) =1

3.0 (hypertension) =0

Exclude 2.0 because pregnancy related only is not comparable to other datasets

7.0 and 9.0 as missing

**_RFSMOK3, we have:**

1.0 =not current smoker

2.0 =current smoker

9.0 =missing/unclear

So,

2.0 =1

1.0 =0

9.0 =missing


**_TOTINDA, we have:**

1.0 =physically active

2.0 =not physically active

9.0 =missing

So,

1.0 =1

2.0 = 0

9.0 =missing














In [ ]:
#Creating a clean BRFSS Dataset
brfss_clean = brfss_sub[[
    "diabetes_binary",
    "age",
    "sex_male",
    "bmi",
    "hypertension",
    "current_smoker",
    "physically_active"
]].copy()

print(brfss_clean.shape)
print(brfss_clean["diabetes_binary"].value_counts(dropna=False))
print(brfss_clean.isna().mean().sort_values())

brfss_clean.to_csv("/content/brfss_clean.csv", index=False)
print ("Saved BRFSS cleaned file to /content/brfss_clean.csv")



(429360, 7)
diabetes_binary
0    372104
1     57256
Name: count, dtype: int64
diabetes_binary      0.000000
age                  0.000000
sex_male             0.000000
hypertension         0.009707
current_smoker       0.040714
bmi                  0.081878
physically_active    0.086345
dtype: float64
Saved BRFSS cleaned file to /content/brfss_clean.csv
